여러 문서에서 찾아서 답변하는 챗봇 만들기
  - QA Chatbot
  - Langchain
  - ChatGPT
  - ChromaDB

In [1]:
# setting langchain
# 설치 후 세션 반드시 다시 시작
# !pip uninstall -y langchain langchain-core langchain-community langchain-openai
!pip install -q langchain langchain-community langchain-core langchain-openai langchain-text-splitters openai tiktoken chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.5/558.5 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/2

In [2]:
# OpenAI API 키를 환경 변수에 설정합니다.
# 실제 API 키는 보안상 직접 코드에 넣지 않고 환경 변수나 secrets manager를 사용하는 것이 좋습니다.
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")

In [3]:
# 기사 21개
!wget -q https://github.com/kairess/toy-datasets/raw/master/techcrunch-articles.zip

In [4]:
# 압축 파일 해제
!unzip -q techcrunch-articles.zip -d articles

Embedding
  - 시스템의 세번째 단계
  - 문서분할 단계에서 생성된 문서 단위들을 기계가 이해할 수 있는 수치적 형태로 변환하는 과정
  - 문서의 의미를 벡터(숫자의 배열)형태로 표시
  - 필요성
    1. 의미 이해
    2. 정보 검색 향상

In [ ]:
!pip install -qU openai langchain langchain-community pypdf tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 28.5 MB/s eta 0:00:00


OpenAIEmbeddings

In [ ]:

# =========================================================
# 1. 필요한 라이브러리
# =========================================================

import os  # 환경 변수 접근용

from langchain_openai import OpenAIEmbeddings  # 최신 embedding import


# =========================================================
# 2. OpenAI API Key 로드
# =========================================================
# 역할: 환경 변수에 저장된 API Key를 안전하게 가져옴

api_key = os.getenv("OPENAI_API_KEY")


# =========================================================
# 3. Embedding 모델 초기화
# =========================================================
# 역할: text → vector 변환 (RAG에서 핵심 단계)
# 결과: 문장을 고차원 숫자 벡터로 변환

embedding_model = OpenAIEmbeddings(
    api_key = api_key
)

In [ ]:

# =========================================================
# 1. Embedding 실행 테스트 목적
# =========================================================
# 역할: 텍스트 → 벡터 변환이 정상적으로 수행되는지 확인
# 의미: RAG에서 chunk → embedding 단계 검증


# =========================================================
# 2. 텍스트 리스트 준비
# =========================================================
# 역할: 한글 + 영어 혼합 데이터를 사용하여 embedding 품질 테스트

texts = [
    "안녕하세요",
    "제 이름은 홍길동입니다",
    "이름이 무엇인가요?",
    "랭체인은 유용합니다",
    "Hello Langchain!"
]


# =========================================================
# 3. Embedding 생성
# =========================================================
# 역할: 각 텍스트를 고차원 vector로 변환
# 결과: (문장 수 x embedding 차원) 구조 생성

embeddings = embedding_model.embed_documents(texts)


# =========================================================
# 4. 결과 shape 확인
# =========================================================
# embeddings 개수 = 문장 개수
# embeddings[0] 길이 = 벡터 차원 (예: 1536)

print("벡터 개수:", len(embeddings))
print("벡터 차원:", len(embeddings[0]))


# =========================================================
# 5. 첫 번째 embedding 출력
# =========================================================

print("\n첫 번째 벡터 샘플:")
print(embeddings[0])

벡터 개수: 5
벡터 차원: 1536

첫 번째 벡터 샘플:
[-0.013639057986438274, -0.009446357376873493, -0.005943919066339731, -0.022944806143641472, -0.012475838884711266, 0.01820245385169983, -0.01977471634745598, 0.005880006123334169, -0.012904057279229164, 0.008788052946329117, 0.012060403823852539, -0.0032292099203914404, -0.024005763232707977, -0.014495492912828922, -0.005608375184237957, -0.007171051111072302, 0.02722698450088501, -0.01342175342142582, 0.007886878214776516, -0.017435496672987938, -0.00429176539182663, -0.01587601564824581, 0.002911242190748453, -0.012469448149204254, 1.4942185771360528e-05, 0.003953025676310062, 0.0040137432515621185, -0.02391628548502922, 0.014201493002474308, -0.021666543558239937, 0.0027019267436116934, 0.006953746546059847, -0.01880323700606823, -0.005582810379564762, 0.010194141417741776, -0.012322447262704372, 0.002403132850304246, -0.014802276156842709, 0.005307983607053757, -0.007439486216753721, 0.018125757575035095, -0.014738363213837147, 0.00928018335253000

In [ ]:
"""

"고양이가 잔다"	  벡터 A
"강아지가 잔다"	  벡터 B
"서울 날씨"	      벡터 C

cosine similarity (가장 중요)
A · B / |A||B|

비교	            결과
고양이 vs 강아지	0.8~0.95 (높음)
고양이 vs 날씨	  0.1~0.3 (낮음)

1.0 → 완전히 같은 의미
0.0 → 전혀 다른 의미
-1.0 → 반대 의미 (거의 안 씀)

즉, 문장의 의미를 좌표화하여 거리를 “비교하는 것”이다

"""

위의 결과를 보면
  - 총 5개의 문장을 임베딩하였고 1536차원의 실수 벡터로 변환되었다
  - 벡터 간 유사도 비교를 통해 검색, 분류, 클러스터링등에 활용할 수 있다



In [ ]:

# =========================================================
# 1. Query / Answer Embedding 테스트 목적
# =========================================================
# 역할: 질문과 답변이 동일한 embedding space에서 얼마나 잘 정렬되는지 확인
# 의미: RAG에서 retrieval 정확도와 직결되는 핵심 테스트


# =========================================================
# 2. 질문(Query) embedding 생성
# =========================================================
# 역할: 사용자의 질문을 vector로 변환

embedded_q = embedding_model.embed_query(
    "이 대화에서 언급된 이름은?"
)


# =========================================================
# 3. 답변(Answer) embedding 생성
# =========================================================
# 역할: 답변 문장을 vector로 변환

embedded_a = embedding_model.embed_query(
    "이 대화에서 언급된 이름은 홍길동."
)


# =========================================================
# 4. embedding 차원 확인
# =========================================================
# 역할: 두 벡터의 차원이 동일한지 확인 (정상 여부 체크)

print("Query embedding 차원:", len(embedded_q))
print("Answer embedding 차원:", len(embedded_a))

Query embedding 차원: 1536
Answer embedding 차원: 1536


In [ ]:
# 유사도 계산
from numpy import dot
from numpy.linalg import norm
import numpy as np

# 벡터 유사도 계산 함수(코사인 유사도)
def cos_sim(A, B):

  return dot(A, B)/(norm(A)*norm(B))

In [ ]:
print(cos_sim(embedded_q, embedded_a))  # 질문과 답변간 유사도
print(cos_sim(embedded_q, embeddings[1]))   # 질문과 '제 이름은 홍길동입니다' 유사도
print(cos_sim(embedded_q, embeddings[3]))   # 질문과 '랭체인은 유용합니다' 유사도

0.9000376025164908
0.8437728310888712
0.7590337997945664


Huggingface Embedding

In [ ]:

# =========================================================
# 1. Sentence Transformers 설치
# =========================================================
# 역할: HuggingFace 기반의 사전학습 임베딩 모델 사용 라이브러리
# 특징: OpenAI 없이 로컬에서도 embedding 생성 가능
# 용도: RAG, semantic search, clustering 등

!pip install -q sentence_transformers

In [ ]:

# =========================================================
# 1. BGE Embedding 모델 임포트
# =========================================================
# 역할: HuggingFace 기반 고성능 embedding 모델 사용
# 특징: OpenAI 없이도 semantic search 가능 (로컬 RAG 핵심)

from langchain_community.embeddings import HuggingFaceBgeEmbeddings


# =========================================================
# 2. 모델 설정
# =========================================================

model_name = "BAAI/bge-small-en"   # 설명: BAAI에서 제공하는 경량 영어 embedding 모델

model_kwargs = {

    "device" : "cpu"
    # GPU 사용 가능하면 "cuda"로 변경
}

encode_kwargs = {
    "normalize_embeddings" : True
    # 역할: cosine similarity 성능 향상을 위해 vector 정규화
}


# =========================================================
# 3. Embedding 모델 초기화
# =========================================================
# 역할: text → dense vector 변환 (RAG 검색 핵심 엔진)

hf = HuggingFaceBgeEmbeddings(
    model_name= model_name,
    model_kwargs = model_kwargs,
    encode_kwargs= encode_kwargs
)

/tmp/ipykernel_1932/2845456338.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceBgeEmbeddings
/tmp/ipykernel_1932/2845456338.py:33: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  hf = HuggingFaceBgeEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/90.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:

# =========================================================
# 1. 테스트용 문장 리스트
# =========================================================
# 역할: 한국어 + 정보성 문장을 넣어 embedding 품질 확인
# 목적: BGE 모델이 한국어 semantic을 얼마나 잘 이해하는지 테스트

sentences = [
    "안녕하세요",
    "제 이름은 홍길동입니다",
    "이름이 무엇인가요?",
    "랭체인은 유용합니다",
    "홍길동 아버지의 이름은 홍상직입니다."
]


# =========================================================
# 2. 문장 → Embedding 변환
# =========================================================
# 역할: BGE 모델을 사용하여 문장을 vector로 변환
# 출력: (문장 수 × embedding 차원) 형태

ko_embeddings = hf.embed_documents(sentences)


# =========================================================
# 3. 결과 확인
# =========================================================

print("임베딩 개수:", len(ko_embeddings))        # 문장 개수
print("벡터 차원:", len(ko_embeddings[0]))      # embedding dimension

# 첫 번째 벡터 출력 (샘플 확인)
print("\n첫 번째 embedding vector:")
print(ko_embeddings[0])

임베딩 개수: 5
벡터 차원: 384

첫 번째 embedding vector:
[-0.0038130481261759996, 0.022688036784529686, 0.024893295019865036, -0.04458450525999069, 0.0005947581375949085, 0.010855820029973984, 0.022047163918614388, 0.0038615704979747534, 0.009037301875650883, -0.018649855628609657, 8.90349292603787e-06, -0.06594862788915634, 0.02349349670112133, 0.024633686989545822, 0.007547575049102306, -0.020361637696623802, 0.007541654631495476, 0.01574847660958767, -0.04895175248384476, 0.006018768530339003, 0.04735436663031578, -0.02440221793949604, 0.004218148998916149, -0.04609302058815956, 0.004514686763286591, 0.027368882670998573, -0.009226072579622269, -0.03371601551771164, -0.034281712025403976, -0.12372750788927078, -0.04077424108982086, -0.039441101253032684, 0.04505784437060356, -0.0015528812073171139, -0.018713688477873802, 0.0025228518061339855, -0.028800565749406815, 0.05218230187892914, -0.037276286631822586, -0.004290982149541378, 0.0346057265996933, -0.021787118166685104, 0.002606212627142667

In [ ]:

# =========================================================
# 1. 질문 문장 embedding 생성
# =========================================================
# 역할: 사용자의 query를 vector space로 변환 (retrieval 기준 벡터)

BGE_q = hf.embed_query(
    "홍길동은 아버지를 아버지라 부르지 못했다. 홍길동의 아버지 이름은?"
)


# =========================================================
# 2. 답변 문장 embedding 생성
# =========================================================
# 역할: 비교 대상 문장을 동일한 vector space로 변환

BGE_a = hf.embed_query(
    "홍길동의 아버지는 엄했다."
)


# =========================================================
# 3. 질문 출력 (context 확인용)
# =========================================================

print(
    "홍길동은 아버지를 아버지라 부르지 못했다. 홍길동의 아버지 이름은?\n",
    "-" * 100
)


# =========================================================
# 4. Query vs Answer 유사도 계산
# =========================================================
# 역할: semantic similarity 확인 (cosine similarity)
# 의미: 값이 높을수록 의미적으로 가까움

print(
    "홍길동의 아버지는 엄했다.\t\t 문장유사도 :",
    round(cos_sim(BGE_q, BGE_a), 2)
)


# =========================================================
# 5. 사전 임베딩된 문장들과 비교
# =========================================================
# 역할: query가 어떤 문장과 더 가까운지 확인 (retrieval simulation)

print(
    sentences[1] + "\t\t\t 문장유사도 :",
    round(cos_sim(BGE_q, ko_embeddings[1]), 2)
)

print(
    sentences[3] + "\t\t\t 문장유사도 :",
    round(cos_sim(BGE_q, ko_embeddings[3]), 2)
)

print(
    sentences[4] + "\t 문장유사도 :",
    round(cos_sim(BGE_q, ko_embeddings[4]), 2)
)

홍길동은 아버지를 아버지라 부르지 못했다. 홍길동의 아버지 이름은?
 ----------------------------------------------------------------------------------------------------
홍길동의 아버지는 엄했다.		 문장유사도 : 0.94
제 이름은 홍길동입니다			 문장유사도 : 0.89
랭체인은 유용합니다			 문장유사도 : 0.84
홍길동 아버지의 이름은 홍상직입니다.	 문장유사도 : 0.92


위의 결과로
  - 홍길동의 아버지는 엄했다.             → 의미 유사도 있음
  - 제 이름은 홍길동입니다                 → 약간 관련 있음
  - 랭체인은 유용합니다                   → 관련 없음
  - 홍길동 아버지의 이름은 홍상직입니다.  → 의미적으로 가장 적절함 (정답에 가까움)

한국어 사전학습 모델 임베딩
  - ko-sbert-nli

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

model_name = 'jhgan/ko-sbert-nli'  # 한국어 사전학습 모델
model_kwargs = {"device":"cpu"}    # 모델 로드 시 사용할 추가 인자
encode_kwargs = {"normalize_embeddings":True}   # 임베딩 정규화 설정

ko = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

/tmp/ipykernel_1932/2961294209.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  ko = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
senteces = [
    '안녕하세요',
    '제 이름은 홍길동입니다',
    '이름이 무엇인가요?',
    '랭체인은 유용합니다',
    '홍길동 아버지의 이름은 홍상직입니다.'
]

# 한글 문장을 BGE 모델로 임베딩
ko_embeddings = ko.embed_documents(senteces)

# 한글 질문과 답변 문장을 BGE 모델로 임베딩
ko_q = ko.embed_query('홍길동은 아버지를 아버지라 부르지 못했다. 홍길동의 아버지 이름은?')
ko_a = ko.embed_query('홍길동의 아버지는 엄했다.')

print('홍길동은 아버지를 아버지라 부르지 못했다. 홍길동의 아버지 이름은? \n', '-'*100)
print('홍길동의 아버지는 엄했다.\t\t 문장유사도 :', round(cos_sim(ko_q, ko_a), 2))
print(senteces[1] + '\t\t\t 문장유사도 :', round(cos_sim(ko_q, ko_embeddings[1]), 2))
print(senteces[3] + '\t\t\t 문장유사도 :', round(cos_sim(ko_q, ko_embeddings[3]), 2))
print(senteces[4] + '\t 문장유사도 :', round(cos_sim(ko_q, ko_embeddings[4]), 2))

홍길동은 아버지를 아버지라 부르지 못했다. 홍길동의 아버지 이름은? 
 ----------------------------------------------------------------------------------------------------
홍길동의 아버지는 엄했다.		 문장유사도 : 0.46
제 이름은 홍길동입니다			 문장유사도 : 0.52
랭체인은 유용합니다			 문장유사도 : 0.03
홍길동 아버지의 이름은 홍상직입니다.	 문장유사도 : 0.59


의미적으로 가까운 문장이 높은 유사도를 보이는지 확인하면, BGE 임베딩과 모델 세팅이 잘 작동하고 있다는 뜻입니다.

특히 홍길동 아버지의 이름은 홍상직입니다. 문장이 높은 점수를 보인다면, 이 시스템은 RAG 기반 검색, 질문-응답 매칭, FAQ 검색, 문서 기반 질의응답 등에 활용 가능성이 높습니다.

한국어 파인 튜닝

In [ ]:

# =========================================================
# 1. HuggingFace Embedding 모델 임포트
# =========================================================
# 역할: HuggingFace 기반 sentence embedding 모델 사용
# 특징: OpenAI 없이도 고성능 semantic search 가능

from langchain_community.embeddings import HuggingFaceEmbeddings


# =========================================================
# 2. 모델 설정
# =========================================================

model_name = "upskyy/bge-m3-korean"
# 설명: 한국어 특화 BGE-M3 파인튜닝 모델
# 특징: multilingual + retrieval optimized

model_kwargs = {
    "device": "cpu"   # GPU 사용 시 "cuda" 권장
}

encode_kwargs = {
    "normalize_embeddings" : True
    # 역할: cosine similarity 안정화 (RAG 필수 옵션)
}


# =========================================================
# 3. Embedding 모델 초기화
# =========================================================
# 역할: 텍스트 → 고차원 vector 변환 (RAG 핵심 엔진)

ko = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
sentences = [
    '안녕하세요',
    '제 이름은 홍길동입니다',
    '이름이 무엇인가요?',
    '랭체인은 유용합니다',
    '홍길동 아버지의 이름은 홍상직입니다.'
]

# 한글 문장을 BGE 모델로 임베딩
ko_embeddings = ko.embed_documents(sentences)

# 한글 질문과 답변 문장을 BGE 모델로 임베딩
ko_q = ko.embed_query('홍길동은 아버지를 아버지라 부르지 못했다. 홍길동의 아버지 이름은?')
ko_a = ko.embed_query('홍길동의 아버지는 엄했다.')

print('홍길동은 아버지를 아버지라 부르지 못했다. 홍길동의 아버지 이름은? \n', '-'*100)
print('홍길동의 아버지는 엄했다.\t\t 문장유사도 :', round(cos_sim(ko_q, ko_a), 2))
print(senteces[1] + '\t\t\t 문장유사도 :', round(cos_sim(ko_q, ko_embeddings[1]), 2))
print(senteces[3] + '\t\t\t 문장유사도 :', round(cos_sim(ko_q, ko_embeddings[3]), 2))
print(senteces[4] + '\t 문장유사도 :', round(cos_sim(ko_q, ko_embeddings[4]), 2))

홍길동은 아버지를 아버지라 부르지 못했다. 홍길동의 아버지 이름은? 
 ----------------------------------------------------------------------------------------------------
홍길동의 아버지는 엄했다.		 문장유사도 : 0.67
제 이름은 홍길동입니다			 문장유사도 : 0.49
랭체인은 유용합니다			 문장유사도 : 0.06
홍길동 아버지의 이름은 홍상직입니다.	 문장유사도 : 0.7


In [ ]:

# =========================================================
# 1. 평가용 문서 데이터 (Corpus)
# =========================================================
# 역할: vector DB 대신 사용할 문서 집합
# 목적: embedding 기반 retrieval 성능 테스트

sentences = [
    "안녕하세요",
    "제 이름은 홍길동입니다",
    "이름이 무엇인가요?",
    "랭체인은 유용합니다",
    "홍길동 아버지의 이름은 홍상직입니다."
]


# =========================================================
# 2. 문서 Embedding 생성
# =========================================================
# 역할: 모든 문서를 vector space로 변환 (retrieval 대상 생성)

doc_embeddings = ko.embed_documents(sentences)


# =========================================================
# 3. Query 정의
# =========================================================
# 역할: 검색 대상 질문 설정

query = "홍길동의 아버지 이름은 무엇인가요?"


# =========================================================
# 4. Query Embedding 생성
# =========================================================
# 역할: 질문을 vector space로 변환 (검색 기준 벡터)

query_embedding = ko.embed_query(query)

# =========================================================
# 5. Cosine Similarity 함수 정의
# =========================================================
# 역할: vector 간 유사도 계산 (RAG 핵심 함수)

import numpy as np

def cos_sim(a, b):

    # numpy array 변환
    a = np.array(a)
    b = np.array(b)

    # cosine similarity 계산
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


# =========================================================
# 6. Top-K Retrieval 수행
# =========================================================
# 역할: query와 모든 document 간 유사도 계산 후 ranking

scores = []

for i, doc_emb in enumerate(doc_embeddings):

    # query vs document similarity 계산
    score = cos_sim(query_embedding, doc_emb)

    # (문서, 점수) 저장
    scores.append((sentences[i], score))


# similarity 기준 내림차순 정렬
scores = sorted(scores, key=lambda x:x[1], reverse=True)


# =========================================================
# 7. Retrieval 결과 출력
# =========================================================
# 역할: embedding 기반 검색 결과 확인

print("\n==============================")
print("QUERY:", query)
print("==============================\n")

print("TOP-K RETRIEVAL RESULTS:\n")

for doc, score in scores:
    print(doc, "\t similarity:", round(score, 3))


# =========================================================
# 8. Recall@K 평가 - retrieval 성능
# =========================================================
# 역할: 실제 정답 문장이 top-k 안에 포함되는지 평가

# Recall@K 평가 형식만 도입
ground_truth = "홍길동 아버지의 이름은 홍상직입니다."

top_k = [doc for doc, _ in scores[:3]]

print("\n==============================")
print("Recall@3:", ground_truth in top_k)
print("==============================")


QUERY: 홍길동의 아버지 이름은 무엇인가요?

TOP-K RETRIEVAL RESULTS:

홍길동 아버지의 이름은 홍상직입니다. 	 similarity: 0.828
제 이름은 홍길동입니다 	 similarity: 0.572
이름이 무엇인가요? 	 similarity: 0.224
안녕하세요 	 similarity: 0.201
랭체인은 유용합니다 	 similarity: 0.138

Recall@3: True


upskyy/bge-m3-korean 모델을 통해 한국어 문장 의미 파악이 잘 되고 있음

특히 정답 문장을 유사도 기반으로 잘 뽑아내는 점에서 RAG 시스템, 검색, FAQ 추천 등에 활용 가능성이 높음

VectorStore
  - 시스템의 네번째 단계로 이전 단계에서 생성된 임베딩 벡터들을 효율적으로 저장하고 관리한다
  - 향후 검색과정에서 벡터들을 빠르게 조회하고, 관련 문서들을 신속하게 찾아내는 데 필수적이다
  - 필요성
    1. 빠른 속도 검색
    2. 스케일러빌리티
    3. 의미 검색


Chroma
  - AI 네이티브 오픈 소스 벡터 데이터 베이

FAISS
  - Facebook AI 유사성검색(FAISS)은 고밀도 벡터의 효율적인 유사성을 검색 및 클러스터링을 위한 라이브러리
  - 모든 크기의 벡터 집합에서 검색하는 알고리즘이 포함되어 있다.

| 항목     | FAISS           | Chroma         |
| ------ | --------------- | -------------- |
| 저장 방식  | 인메모리 (디스크 저장 X) | 디스크 기반, 지속성 있음 |
| 장점     | 빠른 검색, 가볍고 간단   | 쿼리 확장 기능 등 지원  |
| 단점     | 세션 종료 시 데이터 유실  | 약간 더 무거움       |
| 적합한 용도 | 빠른 테스트, 임시 분석   | 프로덕션/장기 운영     |


In [ ]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 62.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

# =========================================================
# 1. FAISS Vector Store 임포트
# =========================================================
# 역할: embedding vector를 저장하고 빠르게 similarity search 수행

from langchain_community.vectorstores import FAISS


# =========================================================
# 2. PDF 로딩
# =========================================================
# 역할: PDF → page 단위 Document 변환

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    "/content/drive/MyDrive/AI_ChatBot/LLM_원본/자료/[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf"
)

pages = loader.load_and_split()


# =========================================================
# 3. Token 기반 Chunk Splitter 설정
# =========================================================
# 역할: PDF를 RAG 최적 크기로 분할

from langchain_text_splitters import RecursiveCharacterTextSplitter
from tiktoken import get_encoding

tokenizer = get_encoding("cl100k_base")

def tiktoken_len(text):

    return len(tokenizer.encode(text))


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,                  # chunk 최대 크기
    chunk_overlap=0,                 # chunk 간 중복 없음
    length_function=tiktoken_len     # token 기준 길이 계산
)


# =========================================================
# 4. Document → Chunk 변환
# =========================================================
# 역할: PDF page를 RAG용 chunk로 변환

docs = text_splitter.split_documents(pages)


# =========================================================
# 5. 한국어 Embedding 모델 초기화
# =========================================================
# 역할: text → vector 변환 (semantic search 핵심)

from langchain_community.embeddings import HuggingFaceEmbeddings


model_name = "jhgan/ko-sbert-nli"

model_kwargs = {
    "device": "cpu"
}

encode_kwargs = {
     "normalize_embeddings":True   # cosine similarity 안정화
}

ko = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)


# =========================================================
# 6. FAISS Vector DB 생성 (핵심 단계)
# =========================================================
# 역할: embedding된 document를 FAISS index에 저장
# 결과: semantic search 가능한 vector database 생성

vectordb = FAISS.from_documents(
    documents=docs,
    embedding=ko
)




Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:

# =========================================================
# 1. FAISS Vector Store 생성
# =========================================================
# 역할: 문서(chunk)를 embedding하여 FAISS index에 저장
# 결과: semantic search 가능한 vector database 생성

db = FAISS.from_documents(
    docs,   # 분할된 문서 (chunk 리스트)
    ko      # 한국어 embedding 모델 (HuggingFaceEmbeddings)
)

In [ ]:
# FAISS + Score 기반 Retrieval
# =========================================================
# 1. 사용자 질문(Query) 정의
# =========================================================
# 역할: vector DB에서 검색할 입력 문장

query = "6대 먹거리 산업은?"


# =========================================================
# 2. FAISS similarity search 실행
# =========================================================
# 역할: query를 embedding → FAISS에서 가장 유사한 문서 검색
# 결과: 가장 관련성 높은 Document 리스트 반환

docs = db.similarity_search(query)


# =========================================================
# 3. 검색 결과 출력
# =========================================================
# 역할: 가장 관련 높은 문서 내용 확인 (RAG context 확인)

print(docs[0].page_content)

정책금융 공급 규모가 매년 약 100% 수준으로 증가하고 있으며, 새정부의 ‘미래 먹거리산업 신성장 전략추진*’에 따라 인공지능 관련 기술로의 금융지원이 늘어날 것으로 전망됨       * 에너지, 방산, 우주항공, 인공지능(AI), 바이오, 탄소중립 대응, 스마트농업을 차세대 6대 먹거리 산업으로 선정


In [ ]:
"""

threshold 설정 가능
score < 0.5 → 사용
score > 1.0 → 무시

"""
# =========================================================
# 1. 사용자 Query 정의
# =========================================================
# 역할: vector DB에서 검색할 질문

query = "6대 먹거리 산업은?"


# =========================================================
# 2. FAISS similarity search + score 반환
# =========================================================
# 역할:
# - query를 embedding으로 변환
# - FAISS에서 유사 문서 검색
# - 각 문서와의 거리/유사도 점수 함께 반환

docs_and_scores = db.similarity_search_with_score(query)


# =========================================================
# 3. 결과 출력
# =========================================================
# 역할: page_content + score만 출력 (metadata 숨김)

for doc, score in docs_and_scores:

    print("=" * 80)
    print("SCORE:", round(score, 4))   # 유사도 점수만 출력
    print(doc.page_content)            # 본문만 출력

SCORE: 1.0591
정책금융 공급 규모가 매년 약 100% 수준으로 증가하고 있으며, 새정부의 ‘미래 먹거리산업 신성장 전략추진*’에 따라 인공지능 관련 기술로의 금융지원이 늘어날 것으로 전망됨       * 에너지, 방산, 우주항공, 인공지능(AI), 바이오, 탄소중립 대응, 스마트농업을 차세대 6대 먹거리 산업으로 선정
SCORE: 1.0596
▶5G 이동통신 시스템 산업의 value chain은 ‘칩셋 및 장비 → 5G 이동통신 단말 및 기지국 → 5G 이동통신 네트워크 → 이동통신 서비스’로 구성되며, 동 산업은 ①전방산업에 대한 파급효과가 큰 산업, ②진입장벽이 높은 산업, ③지속적인 R&D가 요구되는 산업 등의 특징을 가짐￮방송통신 서비스, 사물인터넷, 모바일 뱅킹, 전자상거래, 건설, 에너지, 의료, 국방, 조선, 물류, 자동차 등의 다양한 산업을 대상으로 하는 등 전방산업에 파급효과가 큰 특징이 있음￮초기시장 선점을 통한 높은 진입장벽이 형성되어 독과점 현상이 뚜렷한 산업분야로, 기존 4G LTE 와의 호환성 때문에 5G 이동통신 시스템 시장에서도 기존 사업자와 계약을 진행하는 경향이 있음￮전체 네트워크 설계역량이 경쟁력의 핵심요소이며, 지속적인 제품 개발능력과 고객을 만족시키기 위한 마케팅 활동 등에서 차별적 경쟁우위를 확보하는 것이 중요한
SCORE: 1.0696
자료: 한국전자통신연구원▶스마트센서 산업의 value chain은 ‘센서 재료 및 장비 → 스마트센서 제조 → 응용분야’로 구성되며, 동 산업은 ①다품종 소량 생산의 맞춤형 산업, ②전·후방 파급력이 큰 기반산업, ③융·복합 산업 등의 특징을 가짐￮다품종 소량 생산 구조를 갖고 있으며, 고객 맞춤형 생산이 일반적인 바, 수요 업체와의 협력관계가 중요한 사업화 성공 요인임￮산업 전반에 활용되는 파급력이 큰 산업의 일종이며, 제품의 설계 및 제조를 위해 재료, 기계, 전기전자, 정보통신 등의 다양한 기술이 융합되어야 하는 산업임▶시장조사전문기관 MarketsandMarkets에 따

relevance score 의미

| 값   | 의미    |
| --- | ----- |
| 1.0 | 매우 유사 |
| 0.5 | 보통    |
| 0.0 | 거의 무관 |


In [ ]:
# FAISS 벡터스토어를 로컬디스크에 저장

db.save_local("faiss_index")


In [ ]:
# FAISS 로컬 로드 + 검색
# =========================================================
# 1. 저장된 FAISS Vector DB 로드
# =========================================================
# 역할: 디스크에 저장된 FAISS index를 다시 메모리에 로드
# 중요: embedding 모델(ko)은 반드시 동일해야 함

new_db = FAISS.load_local(
    "faiss_index",
    ko,
    allow_dangerous_deserialization=True
)



# =========================================================
# 2. Query 정의
# =========================================================
# 역할: FAISS에서 검색할 질문

query = "6대 먹거리 산업은?"


# =========================================================
# 3. Similarity Search (Relevance Score 포함)
# =========================================================
# 역할:
# - query embedding 생성
# - FAISS index에서 top-k 검색
# - relevance score 포함 반환

docs = new_db.similarity_search_with_relevance_scores(
    query,
    k=3
)


# =========================================================
# 4. 검색 결과 출력 (metadata 제거)
# =========================================================
# 역할: page_content + score만 출력 (깔끔한 RAG 결과 확인)

print("질문:", query)
print("\n" + "=" * 80 + "\n")

for i, (doc, score) in enumerate(docs):

    print(f"[{i+1}] 유사도 점수: {round(score, 4)}")
    print("-" * 80)

    # 문서 내용만 출력 (metadata 제거)
    print(doc.page_content)

    print("\n")

질문: 6대 먹거리 산업은?


[1] 유사도 점수: 0.25110000371932983
--------------------------------------------------------------------------------
정책금융 공급 규모가 매년 약 100% 수준으로 증가하고 있으며, 새정부의 ‘미래 먹거리산업 신성장 전략추진*’에 따라 인공지능 관련 기술로의 금융지원이 늘어날 것으로 전망됨       * 에너지, 방산, 우주항공, 인공지능(AI), 바이오, 탄소중립 대응, 스마트농업을 차세대 6대 먹거리 산업으로 선정


[2] 유사도 점수: 0.2506999969482422
--------------------------------------------------------------------------------
▶5G 이동통신 시스템 산업의 value chain은 ‘칩셋 및 장비 → 5G 이동통신 단말 및 기지국 → 5G 이동통신 네트워크 → 이동통신 서비스’로 구성되며, 동 산업은 ①전방산업에 대한 파급효과가 큰 산업, ②진입장벽이 높은 산업, ③지속적인 R&D가 요구되는 산업 등의 특징을 가짐￮방송통신 서비스, 사물인터넷, 모바일 뱅킹, 전자상거래, 건설, 에너지, 의료, 국방, 조선, 물류, 자동차 등의 다양한 산업을 대상으로 하는 등 전방산업에 파급효과가 큰 특징이 있음￮초기시장 선점을 통한 높은 진입장벽이 형성되어 독과점 현상이 뚜렷한 산업분야로, 기존 4G LTE 와의 호환성 때문에 5G 이동통신 시스템 시장에서도 기존 사업자와 계약을 진행하는 경향이 있음￮전체 네트워크 설계역량이 경쟁력의 핵심요소이며, 지속적인 제품 개발능력과 고객을 만족시키기 위한 마케팅 활동 등에서 차별적 경쟁우위를 확보하는 것이 중요한


[3] 유사도 점수: 0.24369999766349792
--------------------------------------------------------------------------------
자료: 한국전자

In [ ]:
# Max Marginal Relevance 검색 수행
# -> 이 검색은 가장 유사한 문서를 찾으면서, 결과 문서들간의 다양성 고려
# fetch_k=10 : 유사도 기준으로 먼저 가져온 문서의 총 수
# lambda_mult=0.3 : 다양성을 얼마나 크게 고려할지(0:유사성만, 1:다양성만)

"""

lambda_mult
| 값   | 의미            |
| ---  | -------------   |
| 0.0  | relevance만     |
| 0.3  | balanced (추천) |
| 1.0  | diversity만     |


"""

# =========================================================
# 1. Query 정의
# =========================================================
# 역할: FAISS에서 검색할 질문

query = "6대 먹거리 산업은?"


# =========================================================
# 2. MMR 검색 수행
# =========================================================
# 역할:
# - relevance (유사도) + diversity (다양성) 균형 검색
# - 중복된 내용이 아닌 다양한 context 확보

docs = new_db.max_marginal_relevance_search(
    query,
    k=3,
    fetch_k=10,
    lambda_mult=0.3
)




# =========================================================
# 3. 검색 결과 출력 (metadata 제거)
# =========================================================
# 역할: RAG context 확인용 깔끔 출력

print("질문:", query)
print("\n" + "=" * 80 + "\n")

for i, doc in enumerate(docs):

    print(f"[{i+1}] MMR 검색 결과")
    print("-" * 80)

    # 문서 내용만 출력 (metadata 제거)
    print(doc.page_content)

    print("\n")

질문: 6대 먹거리 산업은?


[1] MMR 검색 결과
--------------------------------------------------------------------------------
정책금융 공급 규모가 매년 약 100% 수준으로 증가하고 있으며, 새정부의 ‘미래 먹거리산업 신성장 전략추진*’에 따라 인공지능 관련 기술로의 금융지원이 늘어날 것으로 전망됨       * 에너지, 방산, 우주항공, 인공지능(AI), 바이오, 탄소중립 대응, 스마트농업을 차세대 6대 먹거리 산업으로 선정


[2] MMR 검색 결과
--------------------------------------------------------------------------------
자료: 한국전자통신연구원▶스마트센서 산업의 value chain은 ‘센서 재료 및 장비 → 스마트센서 제조 → 응용분야’로 구성되며, 동 산업은 ①다품종 소량 생산의 맞춤형 산업, ②전·후방 파급력이 큰 기반산업, ③융·복합 산업 등의 특징을 가짐￮다품종 소량 생산 구조를 갖고 있으며, 고객 맞춤형 생산이 일반적인 바, 수요 업체와의 협력관계가 중요한 사업화 성공 요인임￮산업 전반에 활용되는 파급력이 큰 산업의 일종이며, 제품의 설계 및 제조를 위해 재료, 기계, 전기전자, 정보통신 등의 다양한 기술이 융합되어야 하는 산업임▶시장조사전문기관 MarketsandMarkets에 따르면 세계 스마트센서 시장규모는 2020년 366.5억 달러에서 연평균 19.0% 성장하여 2025년에는 875.8억 달러의 시장을 시현할 것으로 전망됨[세계 스마트센서 시장규모]


[3] MMR 검색 결과
--------------------------------------------------------------------------------
[혁신성장 ICT 산업 정책금융 공급 현황]                                                            (단위: 억 원, 괄

Retriever
  - 정보 검색의 질을 결정
  - 리트리버는 비정형 쿼리가 주어지면 문서를 반환하는 인터페이스이다
  - 벡터 저장소보다 더 일반적이다
  - 문서를 저장할 필요 없이 단지 검색만 할 수 있

FAISS + GPT LCEL 기반 완전 RAG 챗봇

In [ ]:

# =========================================================
# 1. 패키지 설치
# =========================================================
# 역할: RAG 전체 실행에 필요한 최신 LangChain 구성요소 설치

!pip install -q langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu tiktoken



In [ ]:

# =========================================================
# 2. PDF 로딩
# =========================================================
# 역할: PDF → LangChain Document 객체 변환

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('/content/drive/MyDrive/AI_ChatBot/LLM_원본/자료/[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf')

pages = loader.load()


# =========================================================
# 3. 텍스트 청크 분할 (Token-aware)
# =========================================================
# 역할: 긴 문서를 LLM 입력 가능한 chunk 단위로 분리

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # chunk 최대 길이
    chunk_overlap=100      # 의미 보존을 위한 overlap
)

docs = text_splitter.split_documents(pages)


# =========================================================
# 4. Embedding 모델 초기화
# =========================================================
# 역할: text → vector 변환 (semantic search 핵심)

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sbert-nli",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)


# =========================================================
# 5. FAISS Vector DB 생성
# =========================================================
# 역할: embedding된 문서를 vector index로 저장

from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(
    docs,
    embedding
)


# =========================================================
# 6. Retriever 생성
# =========================================================
# 역할: FAISS 기반 similarity search 엔진

retriever = vector_db.as_retriever(search_kwargs={"k":3})

# =========================================================
# 7. LLM 설정 (GPT)
# =========================================================
# 역할: retrieved context 기반 답변 생성

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    temperature=0,
    model="gpt-3.5-turbo"
)


# =========================================================
# 8. Prompt Template 정의
# =========================================================
# 역할: context + question 구조화

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
너는 문서 기반 AI 챗봇이다.

반드시 아래 context만 사용해서 답변해라.

context:
{context}

question:
{question}

답변:
""")


# =========================================================
# 9. LCEL RAG Chain 구성
# =========================================================
# 역할:
# retriever → context 생성 → prompt → LLM → output parsing

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):

    return "\n\n".join([doc.page_content for doc in docs])


rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)


# =========================================================
# 10. RAG 챗봇 실행
# =========================================================
# 역할: 사용자 질문 → retrieval → GPT 답변 생성

query = "6대 먹거리 산업은?"

result = rag_chain.invoke(query)

print(result)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

방송통신 서비스, 사물인터넷, 모바일 뱅킹, 전자상거래, 건설, 에너지, 의료, 국방, 조선, 물류, 자동차 등의 다양한 산업을 대상으로 하는 등 전방산업에 파급효과가 큰 특징이 있습니다.


In [ ]:

# =========================================================
# 1. 필수 패키지 설치
# =========================================================
# 역할: FAISS + LangChain + OpenAI + Tokenizer 전체 설치

!pip install -q langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu tiktoken


In [ ]:
# 문자 수가 아니라 GPT가 실제로 쓰는 token 기준으로 chunk size를 맞추는 함수 사용
"""

PDF
 → Token chunking (tiktoken)
 → Embedding (KoSBERT)
 → FAISS vector DB
 → Retriever
 → LCEL pipeline
 → GPT answer

 """
# =========================================================
# 1. tiktoken 라이브러리 임포트
# =========================================================
# 역할: OpenAI GPT 모델과 동일한 tokenizer 사용
# 특징: 실제 LLM 입력 기준과 동일한 token 단위 계산 가능

import tiktoken


# =========================================================
# 2. tokenizer 초기화
# =========================================================
# 역할: GPT-3.5 / GPT-4 계열에서 사용하는 기본 encoding 로드

tokenizer = tiktoken.get_encoding("cl100k_base")


# =========================================================
# 3. 토큰 길이 계산 함수 정의
# =========================================================
# 역할: 문자열 → token list → token 개수 반환
# 중요: RAG chunk size 기준을 "정확한 LLM 기준"으로 맞춤
# gpt token 기반 chunk로 정확도가 높다

def tiktoken_len(text: str) -> int:

    # 문자열을 token으로 변환
    tokens = tokenizer.encode(text)

    # token 개수 반환
    return len(tokens)

In [ ]:

# =========================================================
# 1. PDF 로딩
# =========================================================
# 역할: PDF → Document 객체 변환

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('/content/drive/MyDrive/AI_ChatBot/LLM_원본/자료/[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf')
pages = loader.load()



# =========================================================
# 2. Token 기반 Text Splitting
# =========================================================
# 역할: LLM context 기준으로 정확한 chunk 생성

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,           # token 기준 chunk size
    chunk_overlap=100,        # 의미 유지용 overlap
    length_function=tiktoken_len
)

docs = text_splitter.split_documents(pages)


# =========================================================
# 3. Embedding 모델 초기화
# =========================================================
# 역할: text → vector 변환 (semantic search)

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sbert-nli",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)


# =========================================================
# 4. FAISS Vector DB 생성
# =========================================================
# 역할: embedding 저장 및 similarity search 엔진

from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(
    docs,
    embedding
)


# =========================================================
# 5. Retriever 생성
# =========================================================
# 역할: FAISS 검색 엔진 (top-k retrieval)

retriever = vector_db.as_retriever(search_kwargs={"k": 3})


# =========================================================
# 6. GPT 모델 설정
# =========================================================
# 역할: retrieved context 기반 답변 생성

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    temperature=0,
    model="gpt-3.5-turbo"
)


# =========================================================
# 7. Prompt Template 정의
# =========================================================
# 역할: context + question 구조화

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
너는 문서 기반 RAG 챗봇이다.

반드시 아래 context만 사용해서 답변해라.

context:
{context}

question:
{question}

답변:
""")


# =========================================================
# 8. context formatting 함수
# =========================================================
# 역할: retrieved Document → 문자열 변환

def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])


# =========================================================
# 9. LCEL RAG Chain 구성
# =========================================================
# 역할:
# retriever → context 생성 → prompt → LLM → output parsing

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)


# =========================================================
# 10. RAG 챗봇 실행
# =========================================================
# 역할: 사용자 질문 → retrieval → GPT 답변 생성

query = "6대 먹거리 산업은?"

result = rag_chain.invoke(query)

print(result)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

방송통신 서비스, 사물인터넷, 모바일 뱅킹, 전자상거래, 건설, 에너지, 의료, 국방, 조선, 물류, 자동차


여러 문서에서 답변찾는 RAG 기반_QA 에이전트
- LangChain의 initialize_agent() 함수를 통해 만들어졌습니다.
이 에이전트는 RAG(Retrieval-Augmented Generation) 기반 툴(search_tool)을 활용해서 질문에 답하는 구조

In [ ]:
"""

시스템 구조도

[API KEY]
   ↓
[LLM + Embedding 모델]
   ↓
[TXT 파일 로딩]
   ↓
[Chunking]
   ↓
[Chroma Vector DB 생성]
   ↓
[Retriever (Top-K 검색)]
   ↓
[Context 생성 (문서 → 문자열)]
   ↓
[Prompt (context + question)]
   ↓
[GPT 실행]
   ↓
[Answer 출력]
   ↓
[무한 질문 루프]

"""

In [ ]:
"""

전체 플로우

[1. ENV SETUP]
    ↓
OpenAI API Key (Colab userdata → env)

────────────────────────────

[2. MODEL LAYER]
    ↓
LLM: gpt-4o-mini
Embedding: text-embedding-3-small

────────────────────────────

[3. DATA LAYER]
    ↓
DirectoryLoader
    ↓
TechCrunch .txt articles
    ↓
Document List

────────────────────────────

[4. CHUNKING LAYER]
    ↓
RecursiveCharacterTextSplitter
    ↓
chunks (800 tokens, overlap 150)

────────────────────────────

[5. VECTOR STORE LAYER]
    ↓
Chroma Vector DB
    ↓
embedding(chunks)
    ↓
persist: ./chroma_store_articles
    ↓
retriever (top-k = 5)

────────────────────────────

[6. CONTEXT PROCESSING LAYER]
    ↓
format_docs()
    ↓
Document → string context

────────────────────────────

[7. PROMPT LAYER]
    ↓
ChatPromptTemplate
    ↓
context + question → structured prompt

────────────────────────────

[8. LCEL RAG PIPELINE (CORE)]
    ↓
RunnablePassthrough (question)
        + retriever | format_docs (context)
    ↓
prompt
    ↓
LLM (gpt-4o-mini)
    ↓
StrOutputParser()
    ↓
FINAL ANSWER

────────────────────────────

[9. INTERACTION LOOP]
    ↓
while True:
    user input
        ↓
rag_chain.invoke(query)
        ↓
answer 출력

"""

In [5]:

# =========================================================
# 1. LangChain + RAG 필수 패키지 설치
# =========================================================
# 역할:
# - LangChain v0.2+ 최신 구조 대응
# - vector DB / text splitter / core 기능 설치

!pip install -qU langchain langchain-core langchain-community langchain-text-splitters


# =========================================================
# 2. Vector DB + LLM + Tokenizer 설치
# =========================================================
# 역할:
# - Chroma: vector database
# - OpenAI: LLM
# - tiktoken: token 기반 chunking

!pip install -qU chromadb openai tiktoken


# =========================================================
# 3. 문서 전처리 라이브러리 설치
# =========================================================
# 역할:
# - PDF / HTML / TXT 등 다양한 문서 파싱
# - 비정형 데이터 처리 지원

!pip install -qU unstructured[local-inference] python-magic tqdm


# =========================================================
# 4. 시스템 레벨 의존성 설치
# =========================================================
# 역할:
# - file type detection (libmagic)

!apt-get install -y libmagic1


# =========================================================
# 5. 데이터셋 다운로드
# =========================================================
# 역할:
# - TechCrunch 기사 데이터셋 다운로드 (RAG 실습용)

!wget -q https://github.com/kairess/toy-datasets/raw/master/techcrunch-articles.zip


# =========================================================
# 6. 데이터 압축 해제
# =========================================================
# 역할:
# - 다운로드된 zip 파일을 articles 폴더에 압축 해제

!unzip -o -q techcrunch-articles.zip -d ./articles > /dev/null

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 23.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 18.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 44.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.7/676.7 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [6]:
# ============================================================
# RAG : 여러 TechCrunch 기사에서 답변 찾기
# ============================================================


# =========================================================
# 1. 라이브러리 설치
# =========================================================
# 역할: RAG 시스템 구성에 필요한 LangChain 전체 패키지 설치
# 구성:
# - core: LCEL 실행 구조
# - community: vectorDB / loader
# - openai: LLM + embedding
# - chromadb: vector store
# - tiktoken: token 기반 chunking

!pip install -qU langchain langchain-core langchain-community langchain-text-splitters langchain-openai chromadb tiktoken



In [8]:

# =========================================================
# 2. API Key 설정
# =========================================================
# 역할: OpenAI API 인증 설정 (Colab 환경 기준)

import os  # 환경변수 설정을 위한 표준 라이브러리

from google.colab import userdata  # Colab Secret storage 접근

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")  # API Key 환경변수 등록


# =========================================================
# 3. 모델 초기화
# =========================================================
# 역할:
# - LLM: 질문 응답 생성
# - Embedding: 문서를 vector로 변환

from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # OpenAI 모델 인터페이스

llm = ChatOpenAI(  # GPT 모델 초기화
    model="gpt-4o-mini",  # 경량 고성능 모델
    temperature=0         # deterministic 출력 (RAG 안정성 확보)
)

embedding = OpenAIEmbeddings(  # embedding 모델 초기화

    model= "text-embedding-3-small"  # 최신 embedding 모델

)


# =========================================================
# 4. 데이터 로딩
# =========================================================
# 역할: TechCrunch 기사 파일 로드 (.txt)

from langchain_community.document_loaders import DirectoryLoader, TextLoader  # 파일 로더

def load_articles(path: str):  # 기사 로딩 함수 정의

    loader = DirectoryLoader(
        path,
        glob= "*.txt",
        loader_cls= TextLoader
    )


    docs = loader.load()  # 문서 리스트 로딩

    print(f"로드된 기사 수: {len(docs)}")  # 데이터 확인 출력

    return docs  # Document 리스트 반환


docs = load_articles("./articles")  # 실제 데이터 로딩 실행


# =========================================================
# 5. Text Chunking
# =========================================================
# 역할: 긴 문서를 LLM 입력 가능한 단위로 분할

from langchain_text_splitters import RecursiveCharacterTextSplitter  # chunk splitter

splitter = RecursiveCharacterTextSplitter(  # recursive splitter 초기화
    chunk_size=800,      # chunk 최대 길이
    chunk_overlap=150    # 문맥 유지용 overlap
)

chunks =  splitter.split_documents(docs)   # Document → chunk 리스트 변환


# =========================================================
# 6. Vector DB 생성 (Chroma)
# =========================================================
# 역할:
# - embedding 저장
# - semantic search 수행

from langchain_community.vectorstores import Chroma  # vector database

vectordb = Chroma.from_documents(  # vector DB 생성
    documents=chunks,              # chunk 데이터 입력
    embedding=embedding,           # embedding 모델
    persist_directory=  "./chroma_store_articles"   # 로컬 저장 경로
)

retriever = vectordb.as_retriever(  # 검색기 생성
       search_kwargs= {"k":5}        # top-5 검색
)


# =========================================================
# 7. Context formatter
# =========================================================
# 역할: 검색된 문서를 LLM 입력 문자열로 변환

def format_docs(docs):  # Document list → string 변환 함수

    return  "\n\n".join([d.page_content for d in docs])  # 문서 합치기


# =========================================================
# 8. Prompt 정의
# =========================================================
# 역할: RAG 답변 구조 정의 (context 기반 강제 응답)

from langchain_core.prompts import ChatPromptTemplate  # prompt template

prompt = ChatPromptTemplate.from_template(  # prompt 생성
    """
너는 TechCrunch 기사 분석 AI이다.

반드시 아래 context만 사용해서 답변해라.

context:
{context}

question:
{question}

답변:
"""
)


# =========================================================
# 9. LCEL RAG Chain 구성
# =========================================================
# 역할:
# Retriever → Context → Prompt → LLM → Output

from langchain_core.runnables import RunnablePassthrough  # 입력 전달
from langchain_core.output_parsers import StrOutputParser  # 문자열 출력 변환

rag_chain = (  # LCEL chain 정의
    {
        "context":  retriever | format_docs,  # 검색 결과 → context 변환
        "question": RunnablePassthrough()      # 입력 질문 그대로 전달
    }
    | prompt   # prompt 적용
    | llm      # GPT 실행
    | StrOutputParser()  # 결과 문자열 변환
)


# =========================================================
# 10. 실행 루프
# =========================================================
# 역할: 사용자 질문 → RAG 답변 생성

print("RAG 챗봇 준비 완료")  # 시스템 준비 메시지 출력
print("RAG 챗봇 준비 완료")
print("예시 질문:")
print(" - What startups received new funding?")
print(" - 최근 인공지능 관련 기사 내용을 요약해줘.")
print(" - Which company was acquired?")


while True:  # 무한 질의 루프 시작

    query = input("\n질문 (exit 입력 시 종료): ")  # 사용자 입력

    if query.lower() in ["exit", "quit", "q"]:  # 종료 조건 체크
        break  # 루프 종료

    answer = rag_chain.invoke(query)  # RAG 실행

    print("\n=== 답변 ===")  # 출력 구분
    print(answer)  # 최종 답변 출력

로드된 기사 수: 21
RAG 챗봇 준비 완료
RAG 챗봇 준비 완료
예시 질문:
 - What startups received new funding?
 - 최근 인공지능 관련 기사 내용을 요약해줘.
 - Which company was acquired?

질문 (exit 입력 시 종료): langgraph에서 상태를 나타내는 클래스가 뭐야?

=== 답변 ===
주어진 context에는 langgraph에서 상태를 나타내는 클래스에 대한 정보가 포함되어 있지 않습니다. 따라서 해당 질문에 대한 답변을 제공할 수 없습니다.

질문 (exit 입력 시 종료): 너 누구랴?

=== 답변 ===
나는 Vint Cerf에 대한 정보를 제공하는 AI로, 그의 기술 혁신과 인터넷 발전에 대한 기여를 다루고 있다. 현재 Google에서 VP 및 수석 인터넷 전도사로 활동하고 있으며, 접근성, 인공지능, 그리고 행성 간 인터넷에 대한 관심을 가지고 있다.

질문 (exit 입력 시 종료): exit


In [ ]:
"""
비교

RAG 구조 :
User → Retriever → VectorDB → LLM → Answer

Agent + RAG 구조 :
User
 ↓
Agent (LLM reasoning)
 ↓
Tool 선택
 ├── RAG Tool (VectorDB 검색)
 ├── Web Tool
 ├── Calculator Tool
 ↓
결과 합성 → Answer

"""